In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "notebooks") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

In [ ]:
import librosa
import torch
import os
import numpy as np

import matplotlib.pyplot as plt
from IPython.display import Audio, display
import random

import torchaudio
#import seaborn as sns
#from transformers import EncodecModel, AutoProcessor
import IPython.display as ipd
from pathlib import Path
import pickle

from transformers import EncodecModel

import tkinter as tk
from tkinter import filedialog

sample_rate=24000

In [ ]:
# Cell 1: Set file path and parameters
# Configure your dataset path and parameters here
file_path = "/slowdisk/esteban/scratchdata/syntex24/data7wav/syn7wav24_tokens_HFdataset_5/train_sm/"  # Update this path
file_path = "/slowdisk/data/estaban_babble_gender/dataset_hf/tokens/"
file_path = "~/slowdisk/data/ProceduralEngineSounds/ECDC/C_full_set_ecdc"
file_path = "~/slowdisk/data/ProceduralEngineSounds/ECDC/C_full_set_ecdc_HF/tokens"


#########   
# If file_name == None, you will be able to choose your file in a graphical "file picker"
#########
file_name = "audio_20251009_180136_6746_alpha_0"  # Update this filename
file_name = None # None to choose with widget


In [ ]:
def pick_file():
    root = tk.Tk()
    root.withdraw()
    return filedialog.askopenfilename(
        title="Select .ecdc file",
        initialdir=file_path,
        filetypes=[("ECDC files", "*.ecdc"), ("All files", "*.*")]
    )

In [ ]:
def load_and_examine_tokens(ecdcpath):
    """Load dataset and examine a specific sample
    
    Returns
    sample, audio_codes, audio_scales, padding_mask
    sample - the data structure containing the record for indexed sample
    audio_codes - [1,8,T] assuming 8 tokens per frame were stored
    audio_scales - probably None since the 24kHz model doesn't normalize by chunk (doesn't chunk at all)
    padding_mask - also not relevant for the 24kHz model
    """

    saved_data = torch.load(ecdcpath)
    
    # Extract the data in the new simplified format
    audio_codes = saved_data['audio_codes']
    audio_scales = saved_data['audio_scales'] 
    audio_length = saved_data['audio_length']
    
    # Create padding mask from the stored audio length
    padding_mask = torch.zeros(1, audio_length, dtype=torch.bool)
    
    print("  ✅ Loaded encoder outputs")
    print(f"  Audio codes shape: {audio_codes.shape}")
    print(f"  Audio scales type: {type(audio_scales)}")
    if isinstance(audio_scales, list):
        print(f"  Audio scales list length: {len(audio_scales)}")
        if len(audio_scales) > 0 and hasattr(audio_scales[0], 'shape'):
            print(f"  Audio scales[0] shape: {audio_scales[0].shape}")
    else:
        print(f"  Audio scales shape: {audio_scales.shape}")
    print(f"  Audio length: {audio_length}")
    print(f"  Padding mask shape: {padding_mask.shape}")
    
    # Handle different possible shapes for display
    if audio_codes.dim() == 4:
        # Shape [1, 1, n_q, time] -> [1, n_q, time] 
        audio_codes = audio_codes.squeeze(1)
    elif audio_codes.dim() == 2:
        # Shape [n_q, time] -> add batch dimension
        audio_codes = audio_codes.unsqueeze(0)
    
    print(f"  Token shape for analysis: {audio_codes.shape}")  # Should be [1, 8, 375] for 6kbps
    print(f"  Number of codebooks: {audio_codes.shape[1]}")
    print(f"  Sequence length: {audio_codes.shape[2]}")
    
    return audio_codes, audio_scales, padding_mask

In [ ]:
def reconstruct_audio_progressive(audio_codes, audio_scales, padding_mask, model, codebook_counts=[2, 4, 8]):
    """Reconstruct audio using different numbers of codebooks"""
    
    reconstructions = {}
    
    for n_codebooks in codebook_counts:
        print(f"\nReconstructing with {n_codebooks} codebooks...")
        
        # Take first n codebooks and ensure batch dimension
        partial_codes = audio_codes[:, :n_codebooks, :].unsqueeze(0)  # Add batch dim
        
        # Ensure all tensors are on the same device as the model
        device = next(model.parameters()).device
        partial_codes = partial_codes.to(device)
        
        if isinstance(audio_scales, list):
            audio_scales_device = [scale.to(device) if hasattr(scale, 'to') else scale for scale in audio_scales]
        else:
            audio_scales_device = audio_scales.to(device) if hasattr(audio_scales, 'to') else audio_scales
            
        padding_mask_device = padding_mask.to(device)
        
        print(f"  Partial codes shape: {partial_codes.shape}")
        
        with torch.no_grad():
            audio_values = model.decode(partial_codes, audio_scales_device, padding_mask_device)[0]
            audio_array = audio_values.squeeze().cpu().numpy()
            
            print(f"  Output shape: {audio_array.shape}")
            print(f"  Audio length: {len(audio_array) / sample_rate:.2f} seconds")
            
            reconstructions[n_codebooks] = audio_array
    
    return reconstructions

# ------------------------------------------------------------------------------------
# one plot per audio in reconstructions

def plot_audio_comparisons(reconstructions):
    """Plot waveforms for different codebook counts"""
    
    n_plots = len(reconstructions)
    fig, axes = plt.subplots(n_plots, 1, figsize=(12, 3*n_plots))
    
    if n_plots == 1:
        axes = [axes]
    
    colors = ['blue', 'red', 'green', 'purple', 'orange']
    
    for i, (n_codebooks, audio) in enumerate(reconstructions.items()):        
        time_axis = np.linspace(0, len(audio) / sample_rate, len(audio))
        
        axes[i].plot(time_axis, audio, color=colors[i % len(colors)], 
                    linewidth=0.5, alpha=0.8)
        axes[i].set_title(f'{n_codebooks} Codebooks')
        axes[i].set_xlabel('Time (seconds)')
        axes[i].set_ylabel('Amplitude')
        axes[i].grid(True, alpha=0.3)
        
        # Add some stats
        rms = np.sqrt(np.mean(audio**2))
        axes[i].text(0.02, 0.95, f'RMS: {rms:.4f}', transform=axes[i].transAxes, 
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------------------------------
# one spectrogram per audio in reconstructions

def plot_spectrograms(reconstructions, sample_rate=24000):
    """Plot spectrograms for different codebook counts"""
    
    n_plots = len(reconstructions)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    
    if n_plots == 1:
        axes = [axes]
    
    for i, (n_codebooks, audio) in enumerate(reconstructions.items()):
        # Compute spectrogram
        D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
        
        # Plot
        img = librosa.display.specshow(D, sr=sample_rate, x_axis='time', y_axis='hz', 
                                     ax=axes[i], cmap='viridis')
        axes[i].set_title(f'{n_codebooks} Codebooks')
        
        # Add colorbar
        plt.colorbar(img, ax=axes[i], format='%+2.0f dB')
    
    fig.suptitle(f'Spectrograms', fontsize=14)
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------------------------------
# One Audio Control for each audio in reconstructions
                  
def create_audio_controls(reconstructions, sample_rate=24000):
    """Create IPython Audio controls for listening"""
    
    print("\n" + "="*60)
    print("🎵 AUDIO PLAYBACK CONTROLS")
    print("="*60)
    
    for n_codebooks, audio in reconstructions.items():
        print(f"\n🎧 {n_codebooks} Codebooks :")
        print(f"   Audio length: {len(audio) / sample_rate:.2f} seconds")
        print(f"   RMS level: {np.sqrt(np.mean(audio**2)):.4f}")
        display(Audio(audio, rate=sample_rate))

In [ ]:
# reconstruct the latents from the codes.
# The original model had a method for doing this, but they hid it for the Hugging Face version
# 
def efficient_codes_to_latents(model, codes):
    """
    Efficient version for repeated use (e.g., in DataLoaders)
    """
    model.eval()
    
    with torch.no_grad():
        # Direct quantizer access with proper transpose
        if codes.shape[0] == 1:  # batch dimension first
            codes_transposed = codes.transpose(0, 1)  # (n_q, batch, time) # Yep, batch second here. Someone broke protocol!
        else:
            codes_transposed = codes
            
        # Direct quantizer decode - this is just lookups + addition
        embeddings = model.quantizer.decode(codes_transposed)
        return embeddings

In [ ]:
# Get the sequence of latents for a sequence of stacked tokens (audio_codes)
# Visualize them as heat maps

def analyze_128d_latents(audio_codes, audio_scales, padding_mask, model, n_codebooks=8, sample=None):
    """
    Reconstruct 128-D latent vectors from tokens and create waterfall plot
    
    Args:
        audio_codes: Token tensor [1, 8, time]
        model: EnCodec model
        n_codebooks: Number of codebooks to use
        sample: Sample metadata for title
    """
    
    print(f"\nAnalyzing 128-D latents with {n_codebooks} codebooks...")
    
    # Take first n codebooks
    partial_codes = audio_codes[:, :n_codebooks, :]  # [1, n, time]
    
    # Ensure tensor is on the same device as the model
    device = next(model.parameters()).device
    partial_codes = partial_codes.to(device)

    with torch.no_grad():
        # HF different from CodeCraft - no decode_talent method!!!
        # latent_embeddings = model.decode_latent(codes) # What we'd really like
        
        latent_embeddings = efficient_codes_to_latents(model, partial_codes)
        print(f' ==========   SHAPE of latent_embeddings is {latent_embeddings.shape}')
        
        # Remove batch dimension and convert to numpy
        latents = latent_embeddings.squeeze(0).cpu().numpy()  # [128, time]
        
    print(f"  Latent embeddings shape: {latents.shape}")
    print(f"  Time frames: {latents.shape[1]}")
    print(f"  Latent dimensions: {latents.shape[0]}")
    
    # Create waterfall plot
    plt.figure(figsize=(15, 8))
    
    # Plot as heatmap/waterfall
    im = plt.imshow(latents, aspect='auto', cmap='RdBu_r', interpolation='nearest')
    
    # Customize plot
    title = f'128-D Latent Vectors Over Time ({n_codebooks} codebooks)'
    if sample:
        title += f' - {sample["class_name"]}'
    plt.title(title, fontsize=14)
    
    plt.xlabel('Time Frame')
    plt.ylabel('Latent Dimension')
    plt.colorbar(im, label='Magnitude')
    
    # Add some statistics
    mean_magnitude = np.mean(np.abs(latents))
    max_magnitude = np.max(np.abs(latents))
    plt.text(0.02, 0.98, f'Mean |magnitude|: {mean_magnitude:.3f}\nMax |magnitude|: {max_magnitude:.3f}', 
             transform=plt.gca().transAxes, verticalalignment='top',
             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    # Plot magnitude over time (RMS across latent dimensions)
    plt.figure(figsize=(12, 4))
    rms_over_time = np.sqrt(np.mean(latents**2, axis=0))  # RMS across 128 dimensions for each time frame
    time_axis = np.arange(len(rms_over_time))
    
    plt.plot(time_axis, rms_over_time, 'b-', linewidth=1.5, alpha=0.8)
    plt.title(f'RMS Magnitude of 128-D Latents Over Time ({n_codebooks} codebooks)')
    plt.xlabel('Time Frame')
    plt.ylabel('RMS Magnitude')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Show some statistics
    print(f"\n📊 Latent Statistics:")
    print(f"  Mean magnitude: {mean_magnitude:.4f}")
    print(f"  Max magnitude: {max_magnitude:.4f}")
    print(f"  Min magnitude: {np.min(np.abs(latents)):.4f}")
    print(f"  Std magnitude: {np.std(latents):.4f}")
    print(f"  Dynamic range: {max_magnitude/np.mean(np.abs(latents)):.2f}x")
    
    return latents

In [ ]:
# Setup model
print("Loading EnCodec model...")
model = EncodecModel.from_pretrained("facebook/encodec_24khz")
model.config.target_bandwidths = [6.0]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using device: {device}")

print(f"Model training mode: {model.training}")

In [ ]:

    
if file_name == None:
    file_path = pick_file()
    if file_path:
        full_path = Path(file_path)
        print(f"Selected: {full_path}")
    else:
        print("No file selected")
        # Fallback to your current method
        file_path = "/path/to/your/dataset"
        file_name = "example.ecdc"
        full_path = Path(file_path) / file_name

load_and_examine_tokens(full_path)

In [ ]:
audio_codes, audio_scales, padding_mask = load_and_examine_tokens(full_path)

# Move data to same device as model  
audio_codes = audio_codes.to(device)
if isinstance(audio_scales, list):
    audio_scales = [scale.to(device) if hasattr(scale, 'to') else scale for scale in audio_scales]
elif hasattr(audio_scales, 'to'):
    audio_scales = audio_scales.to(device)
padding_mask = padding_mask.to(device)

In [ ]:
# Reconstruct with different codebook counts
print(f"\n🔊 Reconstructing audio...")
reconstructions = reconstruct_audio_progressive(audio_codes, audio_scales, padding_mask, model, [2, 4, 8])#[2, 4, 8])

# Plot comparisons
print(f"\n📈 Plotting waveform comparisons...")
plot_audio_comparisons(reconstructions)

In [ ]:
    print(f"\n📊 Plotting spectrograms...")
    plot_spectrograms(reconstructions)
    
    # Audio controls
    create_audio_controls(reconstructions)